**Notebook overview**
- Purpose: Download NYT monthly archive and extract conflict-related headlines.
- Produces: raw monthly JSON under `data/raw/` and standardized CSV(s) under `data/processed/` with schema: `date`, `news source`, `title`, `link`.
- Notes: Processing includes keyword filtering and duplicate removal.



In [1]:
# NYT: function to fetch monthly archive and filter conflict headlines
from datetime import datetime
import json
import pandas as pd
import requests
from pathlib import Path

# Define your core action signal and infrastructure keywords
CONFLICT_KEYWORDS = [
    "missile",
    "strike",
    "explosion",
    "drone",
    "barrage",
    "blast",
    "attack",
    "bombardment",
    "shelling",
    "kamikaze",
    "shahed",
    "blackout",
    "power outage",
    "energy",
    "electricity",
    "grid",
    "infrastructure",
    "attack",
    "attacked",
    "attacks",
    "strike",
    "struck",
    "strikes",
    "striked",
    "explosion",
    "exploded",
    "explodes",
    "explode",
    "blast",
    "blasted",
    "bomb",
    "bombed",
    "bombing",
    "killed",
    "killing",
    "injured",
    "injury",
    "hit",
    "hits",
    "shelling",
    "bombardment",
    "burning",
    "burned",
    "fire",
    "wildfire",
    "forest fire",
    "forestfire",
    "fire",
    "brush fire",
    "bushfire",
    "blaze",
    "inferno",
    "arson",
    "flames",
    "burning",
    "smoke",
    "spread",
    "spreaded",
    "wind",
    "burn",
    "burned",
]


def get_nyt_archive_headlines(year, month, api_key, country = None):
    """Download one NYT monthly archive, filter conflict headlines, and save processed output."""
    print("==================================================")
    print(f"📰 Accessing NYT Bulk Archive Stream for: {year}-{month:02d}")
    print("==================================================")

    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    params = {"api-key": api_key}

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 429:
            print("❌ Rate Limit Triggered. Please wait before running again.")
            return pd.DataFrame()
        if response.status_code != 200:
            print(f"❌ Failed to extract archive. HTTP Status: {response.status_code}")
            return pd.DataFrame()

        data = response.json()
        all_articles = data.get("response", {}).get("docs", [])
        print(f"📦 Successfully unpacked {len(all_articles)} total monthly articles.")

        # Save the raw API response to News/data/raw as readable JSON
        news_root = Path.cwd()
        if news_root.name.lower() != "news" and (news_root / "News").exists():
            news_root = news_root / "News"
        raw_dir = news_root / "data" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = raw_dir / f"nyt_raw_{year}{month:02d}.json"
        with raw_path.open("w", encoding="utf-8") as fh:
            json.dump(data, fh, ensure_ascii=False, indent=2, default=str)
        print(f"💾 Raw NYT archive saved: {raw_path}")

    except Exception as e:
        print(f"❌ Network connection failed: {e}")
        return pd.DataFrame()

    filtered_headlines = []
    for doc in all_articles:
        pub_date_str = doc.get("pub_date", "")
        if not pub_date_str:
            continue
        try:
            dt = datetime.strptime(pub_date_str[:10], "%Y-%m-%d")
        except Exception:
            continue

        headline_main = doc.get("headline", {}).get("main", "")
        abstract_text = doc.get("abstract", "")
        text_pool = f"{headline_main} {abstract_text} {doc.get('lead_paragraph', '')}".lower()
        if country.lower() in text_pool:
            has_keyword = any(kw in text_pool for kw in CONFLICT_KEYWORDS)
            if has_keyword and headline_main:
                filtered_headlines.append(
                    {
                        "source": "The New York Times",
                        "country": country,
                        "date": dt.strftime("%Y-%m-%d"),
                        "headline": headline_main,
                        "url": doc.get("web_url", ""),
                    }
                )

    if filtered_headlines:
        df_nyt = pd.DataFrame(filtered_headlines)
        df_nyt.drop_duplicates(subset=["headline"], inplace=True)
        processed_dir = Path("data") / "processed"
        processed_dir.mkdir(parents=True, exist_ok=True)
        processed_path = processed_dir / f"nyt_{year}{month:02d}.csv"
        df_nyt.to_csv(processed_path, index=False, encoding="utf-8")
        print(f"💾 Processed NYT conflict news saved: {processed_path}")
        print(f"✅ Slicing Success! Recovered {len(df_nyt)} clean conflict headlines.")
        return df_nyt

    print("⚠️ No headlines matched your criteria within the selected month.")
    return pd.DataFrame()


In [55]:
# --- Execution Configuration ---
MY_NYT_KEY = "8zqUF4rs33orAdv96UGfJaIi9VaYeWHAurOgFrXaPnRWMtmw"  # Paste your API key here

# Target month: October 2022
TARGET_YEAR = 2026
TARGET_MONTH = 6
COUNTRY = ""

# --- Execute Data Harvesting ---
df_nyt_final = get_nyt_archive_headlines(
    year=TARGET_YEAR,
    month=TARGET_MONTH,
    api_key=MY_NYT_KEY,
    country=COUNTRY,
)

# --- View the Collected Headline Rows ---
df_nyt_final.head()


📰 Accessing NYT Bulk Archive Stream for: 2026-06
❌ Failed to extract archive. HTTP Status: 403


""


In [56]:
# --- Process raw NYT archive JSON(s) -> standardized CSV(s) ---
import json
import re
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data") / "raw"
PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

CONFLICT_KEYWORDS = [
    "missile",
    "strike",
    "explosion",
    "drone",
    "barrage",
    "blast",
    "attack",
    "bombardment",
    "shelling",
    "kamikaze",
    "shahed",
    "blackout",
    "power outage",
    "energy",
    "electricity",
    "grid",
    "infrastructure",
    "missile",
    "strike",
    "explosion",
    "drone",
    "barrage",
    "blast",
    "attack",
    "bombardment",
    "shelling",
    "kamikaze",
    "shahed",
    "blackout",
    "power outage",
    "energy",
    "electricity",
    "grid",
    "infrastructure",
    "attack",
    "attacked",
    "attacks",
    "strike",
    "struck",
    "strikes",
    "striked",
    "explosion",
    "exploded",
    "explodes",
    "explode",
    "blast",
    "blasted",
    "bomb",
    "bombed",
    "bombing",
    "killed",
    "killing",
    "injured",
    "injury",
    "hit",
    "hits",
    "shelling",
    "bombardment",
    "burning",
    "burned",
    "fire",
    "wildfire",
    "forest fire",
    "forestfire",
    "fire",
    "brush fire",
    "bushfire",
    "blaze",
    "inferno",
    "arson",
    "flames",
    "burning",
    "smoke",
    "spread",
    "spreaded",
    "wind",
    "burn",
    "burned",
]


def parse_datetime(value):
    if value is None or value == "":
        return pd.NaT
    text = str(value).strip()
    parsed = pd.to_datetime(text, errors="coerce")
    if pd.isna(parsed):
        return pd.NaT
    if not any(token in text.lower() for token in [":", "am", "pm", "t"]):
        parsed = parsed.normalize() + pd.Timedelta(hours=12)
    return parsed


def matches_keywords(text):
    lowered = (text or "").lower()
    return any(keyword in lowered for keyword in CONFLICT_KEYWORDS)


def combine_headline_snippet_paragraph(headline, snippet, lead_paragraph):
    headline_text = re.sub(r"\s+", " ", str(headline or "")).strip()
    snippet_text = re.sub(r"<[^>]+>", " ", str(snippet or ""))
    snippet_text = re.sub(r"\s+", " ", snippet_text).strip()
    lead_text = re.sub(r"\s+", " ", str(lead_paragraph or "")).strip()

    parts = [part for part in [headline_text, snippet_text, lead_text] if part]
    combined = " - ".join(parts)
    if combined:
        return combined
    return headline_text or snippet_text or lead_text


for raw_file in sorted(RAW_DIR.glob("nyt_raw_*.json")):
    try:
        obj = json.loads(raw_file.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Failed to load {raw_file}: {exc}")
        continue

    docs = obj.get("response", {}).get("docs", [])
    rows = []
    for doc in docs:
        headline_obj = doc.get("headline")
        if isinstance(headline_obj, dict):
            headline = headline_obj.get("main", "")
        else:
            headline = headline_obj or ""
        snippet = doc.get("snippet", "")
        lead_paragraph = doc.get("lead_paragraph", "")
        title = combine_headline_snippet_paragraph(headline, snippet, lead_paragraph)
        link = doc.get("web_url", "")
        if not title or not link:
            continue

        text_pool = " ".join(
            [
                str(headline),
                str(snippet),
                str(lead_paragraph),
                str(doc.get("abstract", "")),
            ]
        )
        if not matches_keywords(text_pool):
            continue

        rows.append(
            {
                "date": parse_datetime(doc.get("pub_date", "")),
                "news source": "NYT",
                "title": title,
                "link": link,
            }
        )

    if not rows:
        print(f"No matching articles found in {raw_file}")
        continue

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["date", "title", "link"])
    df = df.drop_duplicates(subset=["link"]).reset_index(drop=True)
    df = df[["date", "news source", "title", "link"]]

    processed_name = raw_file.name.replace("_raw", "")
    processed_path = PROC_DIR / processed_name.replace(".json", ".csv")
    df.to_csv(processed_path, index=False, encoding="utf-8")
    print(f"Saved processed NYT to: {processed_path} ({len(df)} rows)")


Saved processed NYT to: data\processed\nyt_202601.csv (581 rows)
Saved processed NYT to: data\processed\nyt_202602.csv (536 rows)
Saved processed NYT to: data\processed\nyt_202603.csv (934 rows)
Saved processed NYT to: data\processed\nyt_202604.csv (749 rows)
Saved processed NYT to: data\processed\nyt_202605.csv (379 rows)


**What this notebook does**
- Downloads the NYT monthly archive for the target month and saves the raw response under `data/raw/`.
- Re-processes each saved raw archive into a standardized CSV under `data/processed/`.

**Cleaning process**
- Read `response.docs` from the raw JSON archive.
- Keep only the shared analysis columns: `date`, `news source`, `title`, `link`.
- Parse `pub_date` into a full datetime and default the time to 12:00 when only a date is available.
- Set the news source label to `NYT`.
- Apply keyword filtering again during processing so the final CSV keeps only relevant articles.
- Remove duplicate rows by `link` and save the cleaned CSV with `_raw` removed from the filename.

---
`Merging all data`


In [57]:
# Incremental merge for NYT: keep history in nyt_all.csv and append newcomers
from pathlib import Path
import pandas as pd

PROC_DIR = Path("data") / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROC_DIR / "nyt_all.csv"

nyt_files = sorted(
    f for f in PROC_DIR.glob("nyt_*.csv")
    if f.name.lower() != "nyt_all.csv"
)

frames = []

if out_path.exists():
    try:
        existing_all = pd.read_csv(out_path)
        frames.append(existing_all)
        print(f"Loaded existing master {out_path.name}: {len(existing_all)} rows")
    except Exception as exc:
        print(f"Skipped existing master due to read error: {exc}")

for f in nyt_files:
    try:
        df = pd.read_csv(f)
        frames.append(df)
        print(f"Loaded {f.name}: {len(df)} rows")
    except Exception as exc:
        print(f"Skipped {f.name} due to read error: {exc}")

if not frames:
    print(f"No NYT data found in: {PROC_DIR.resolve()}")
else:
    merged = pd.concat(frames, ignore_index=True)

    preferred_cols = ["date", "news source", "title", "link"]
    existing_preferred = [c for c in preferred_cols if c in merged.columns]
    if existing_preferred:
        merged = merged[existing_preferred + [c for c in merged.columns if c not in existing_preferred]]

    if "link" in merged.columns:
        merged = merged.drop_duplicates(subset=["link"], keep="first")
    elif all(c in merged.columns for c in ["title", "date"]):
        merged = merged.drop_duplicates(subset=["title", "date"], keep="first")
    else:
        merged = merged.drop_duplicates()

    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved incremental NYT master: {out_path.resolve()} ({len(merged)} rows)")

merged.head() if 'merged' in locals() else None

Loaded existing master nyt_all.csv: 22939 rows
Loaded nyt_202601.csv: 581 rows
Loaded nyt_202602.csv: 536 rows
Loaded nyt_202603.csv: 934 rows
Loaded nyt_202604.csv: 749 rows
Loaded nyt_202605.csv: 379 rows
Loaded nyt_last_30_days.csv: 40 rows
Saved incremental NYT master: C:\.Okul\CE49X Introduction to Computational Thinking and Data Science for Civil Engineers\GithubRepository\CE49X\Final Project\News\data\processed\nyt_all.csv (22939 rows)


,date,news source,title,link,source,country,headline,url
0,2022-10-01 00:40:53+00:00,NYT,"North Korea Launches Two Ballistic Missiles, F...",https://www.nytimes.com/2022/09/30/world/asia/...,NaN,NaN,NaN,NaN
1,2022-10-01 02:00:03+00:00,NYT,Snapping the Fingers - Natan Last drops off a ...,https://www.nytimes.com/2022/09/30/crosswords/...,NaN,NaN,NaN,NaN
2,2022-10-01 04:01:10+00:00,NYT,Safety Concerns Overshadow Europe’s First New ...,https://www.nytimes.com/2022/10/01/business/ba...,NaN,NaN,NaN,NaN
3,2022-10-01 08:35:31+00:00,NYT,How Do You Stop Erling Haaland? You Don’t. - T...,https://www.nytimes.com/2022/10/01/sports/socc...,NaN,NaN,NaN,NaN
4,2022-10-01 09:00:24+00:00,NYT,Fashion Week Comes Home - Nineteen days and mo...,https://www.nytimes.com/2022/10/01/style/paris...,NaN,NaN,NaN,NaN
